# Metajeux-06 : Le chemin minimal sur un second substrat — jeux ordinaux 3×2, murs traversables et raccourcis

**Série** : GameTheory / Metajeux — jumeau Rust du carnet Python
**Source du portage** : [Metajeux-06-Chemin-Minimal-3x2-Ordinal-Python.ipynb](Metajeux-06-Chemin-Minimal-3x2-Ordinal-Python.ipynb)
**Kernel** : `rust` (evcxr, `std` seul) — mêmes témoins recalculés : nombres de Fubini, recensement 3×2, chemins certifiés, géodésiques

**Navigation** : [Index](README.md) | [<< Metajeux-05-Temoins-Impossibilite](Metajeux-05-Temoins-Impossibilite-Python.ipynb) | [Metajeux-06 Python — source du portage](Metajeux-06-Chemin-Minimal-3x2-Ordinal-Python.ipynb)

> **Seconde attestation, second substrat** : la première ([Metajeux-04](Metajeux-04-Chemin-Minimal-Robinson-Goforth-Python.ipynb), ex. 1-4, et [Metajeux-05](Metajeux-05-Temoins-Impossibilite-Python.ipynb), ex. A-C) vivait sur les jeux 2×2 ordinaux de Robinson-Goforth ; celle-ci monte au substrat **3×2 ordinal** — chaque joueur range 6 cases, l'univers des chambres strictes passe de 24×24 = 576 à 720×720 = 518 400 jeux.

**La question falsifiable que ce notebook pose.** Sur 2×2, GT-20 a mesuré que traverser les murs (passer par des ex æquo) ne raccourcit **jamais** un chemin minimal entre chambres : 0 raccourci sur 576 paires. Ce verdict survive-t-il au changement de substrat ? La réponse mesurée ici est **non** : sur 3×2, 77 760 paires sur 518 400 (**15,0 %**) raccourcissent en traversant les murs, et le diamètre par côté se contracte de 15 à 10. La stratification qui ne comprimait pas la métrique à 2×2 **devient une compression métrique** à 3×2.

**La loi que ce notebook met à l'épreuve** — Loi II du deuxième voyage de digestion ICT :

> *recoordonner + passer du vérificateur au constructeur* : le précédent important n'est pas seulement le changement de coordonnées ; c'est changement de coordonnées **plus** passage du vérificateur au constructeur.

Comme GT-20, chaque témoin est produit par un **constructeur** (BFS + remontée des parents) puis **certifié par un vérificateur séparé** qui ne réutilise rien : il re-dérive l'adjacence par son propre code et re-fait une recherche exhaustive pour la minimalité.

**Plan.** §1 les deux univers (ordres faibles, chambres) ; §2 les générateurs et les trois modèles de graphe ; §3 reproduction du verdict 2×2 sous modèle fermé ; §4 le recensement exhaustif 3×2 — la réfutation ; §5 le produit de jeux et son lemme de décomposition ; §6 le constructeur et le vérificateur séparés (Loi II), y compris le chemin de l'antipode **à travers les murs** ; §7 le contrôle négatif ; §8 les géodésiques ; §9 trois exercices.

In [2]:
// === Configuration : GameTheory 20c, chemin minimal 3x2 ordinal - seconde attestation de l'operation 13 ===
// Imports Python equivalents : std couvre tout (std::time::Instant pour time,
// boucles odometre pour itertools.product, next_comb pour itertools.combinations,
// std::collections::VecDeque pour collections.deque).

println!("GameTheory 20c : chemin minimal sur jeux ordinaux 3x2 (seconde attestation, operation 13)");
println!("Representation : jeu = (table_ligne, table_colonne), chaque table range 6 cases, rangs {{1..6}}");
println!("Convention GT-20 conservee : le rang 6 est le MEILLEUR resultat possible");
println!("Loi II : le constructeur PRODUIT le chemin, le verificateur separe le CERTIFIE par re-derivation");


GameTheory 20c : chemin minimal sur jeux ordinaux 3x2 (seconde attestation, operation 13)


Representation : jeu = (table_ligne, table_colonne), chaque table range 6 cases, rangs {1..6}


Convention GT-20 conservee : le rang 6 est le MEILLEUR resultat possible


Loi II : le constructeur PRODUIT le chemin, le verificateur separe le CERTIFIE par re-derivation


### Lecture du résultat — le changement d'échelle comme variable expérimentale : un seul paramètre change

Le changement d'échelle est la variable expérimentale : GT-20 étudiait des tables 2×2 (4 cases par joueur), ici chaque joueur ordinal dispose d'une table **3×2** — 6 cases à ranger. Toute la machinerie conceptuelle (chambres strictes, murs d'ex æquo, facettes) est portée telle quelle ; c'est précisément ce qui rend la comparaison falsifiable : *un seul paramètre change, le substrat*. La convention des rangs suit GT-20 (`6` = meilleur), et la Loi II structure tout le notebook : aucun témoin n'est accepté sur la parole du code qui l'a produit.

In [3]:
// === Section 1.1 : enumeration des ordres faibles sur 6 cases (et rappel 4 cases) ===

fn canonique(t: &[u8]) -> Vec<u8> {
    // Cle canonique d'un uplet : relabelage croissant des valeurs distinctes (GT-20, portee tel quel).
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    t.iter().map(|v| vals.iter().position(|w| w == v).unwrap() as u8 + 1).collect()
}

fn ordres_faibles(n: u8) -> Vec<Vec<u8>> {
    // Tous les rangements faibles (avec ex aequo) de n cases, formes canoniques.
    // Odometre sur les n^n uplets en ordre lexicographique (= ordre d'itertools.product),
    // canonisation, dedoublonnage : l'iteration du BTreeSet EST le sorted({...}) de Python.
    let mut set = std::collections::BTreeSet::new();
    let mut t = vec![1u8; n as usize];
    let total = (n as u64).pow(n as u32);
    for _ in 0..total {
        set.insert(canonique(&t));
        let mut i = n as usize;
        while i > 0 {
            i -= 1;
            if t[i] < n {
                t[i] += 1;
                break;
            }
            t[i] = 1;
        }
    }
    set.into_iter().collect()
}

fn nb_distinctes(t: &[u8]) -> usize {
    let mut s = t.to_vec();
    s.sort();
    s.dedup();
    s.len()
}

let of4 = ordres_faibles(4);
let st4: Vec<Vec<u8>> = of4.iter().filter(|t| nb_distinctes(t) == 4).cloned().collect();
let of6 = ordres_faibles(6);
let st6: Vec<Vec<u8>> = of6.iter().filter(|t| nb_distinctes(t) == 6).cloned().collect();

println!("Substrat 2x2 (GT-20)  : ordres faibles = {} (nombre de Fubini F4) | stricts = {} | chambres = {}",
    of4.len(), st4.len(), st4.len() * st4.len());
println!("Substrat 3x2 (ici)    : ordres faibles = {} (nombre de Fubini F6) | stricts = {} | chambres = {}",
    of6.len(), st6.len(), st6.len() * st6.len());
println!("Avec ex aequo 3x2     : {} tables liees -- les futures faces des murs", of6.len() - st6.len());


Substrat 2x2 (GT-20)  : ordres faibles = 75 (nombre de Fubini F4) | stricts = 24 | chambres = 576


Substrat 3x2 (ici)    : ordres faibles = 4683 (nombre de Fubini F6) | stricts = 720 | chambres = 518400


Avec ex aequo 3x2     : 3963 tables liees -- les futures faces des murs


### Lecture du résultat — les nombres de Fubini : 720 tables strictes, 4 683 ordres faibles, 518 400 chambres

Les nombres de Fubini comptent les ordres faibles : F4 = 75, F6 = 4 683. Le côté 3×2 porte donc 720 ordres stricts (= 6!) et 4 683 ordres faibles, contre 24 (= 4!) et 75 pour le 2×2. Le produit des deux joueurs donne l'univers des jeux : 518 400 chambres strictes en 3×2, 21 930 489 jeux au total si l'on admet les murs des deux côtés à la fois — un univers que nous ne parcourrons pas exhaustivement au niveau jeu (voir le lemme du §5, qui le couvre par décomposition). L'explosion combinatoire est le prix du changement de substrat ; elle est aussi ce qui rend la question des raccourcis non triviale.

In [4]:
// === Section 1.2 : les jeux canoniques 3x2 (convention GT-21 / GT-3b / GT-20 etendue) ===

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

fn nb_distinctes(t: &[u8]) -> usize {
    let mut s = t.to_vec();
    s.sort();
    s.dedup();
    s.len()
}

fn afficher_jeu(nom: &str, jeu: &[Vec<u8>]) {
    let (row, col) = (&jeu[0], &jeu[1]);
    println!("{}", nom);
    for i in 0..3 {
        println!("  Ligne | {:>2}  {:<2}|      Colonne | {:>2}  {:<2}|",
            row[2 * i], row[2 * i + 1], col[2 * i], col[2 * i + 1]);
    }
}

let identite: Vec<Vec<u8>> = vec![vec![1, 2, 3, 4, 5, 6], vec![1, 2, 3, 4, 5, 6]];    // les deux joueurs preferent la meme case
let renverse: Vec<Vec<u8>> = vec![vec![6, 5, 4, 3, 2, 1], vec![6, 5, 4, 3, 2, 1]];    // les deux antipodes du permutoedre S6
let rotation: Vec<Vec<u8>> = vec![vec![3, 4, 5, 6, 1, 2], vec![5, 6, 1, 2, 3, 4]];    // conflit pur : chaque meilleure case d'un joueur
let mur_partiel: Vec<Vec<u8>> = vec![vec![1, 2, 2, 3, 4, 5], vec![1, 2, 3, 4, 5, 6]]; // Ligne indifferente entre ses cases 2 et 3

for (nom, j) in [("Identite", &identite), ("Renversement", &renverse), ("Rotation", &rotation),
                 ("Mur partiel (Ligne indifferente 2~3)", &mur_partiel)] {
    afficher_jeu(nom, j);
    let strict = nb_distinctes(&j[0]) == 6 && nb_distinctes(&j[1]) == 6;
    println!("  strict (chambre) : {} | mur : {}", py_bool(strict), py_bool(!strict));
    println!();
};


Identite


  Ligne |  1  2 |      Colonne |  1  2 |


  Ligne |  3  4 |      Colonne |  3  4 |


  Ligne |  5  6 |      Colonne |  5  6 |


  strict (chambre) : True | mur : False


Renversement


... (10 outputs left out)

  Ligne |  1  2 |      Colonne |  1  2 |


  Ligne |  2  3 |      Colonne |  3  4 |


  Ligne |  4  5 |      Colonne |  5  6 |


  strict (chambre) : False | mur : True


### Lecture du résultat — les jeux canoniques : deux antipodes, un conflit d'intérêts, et `MUR_PARTIEL` hors des chambres

L'`Identite` et le `Renversement` sont les deux antipodes du permutoèdre S6 × S6 : leur distance par côté est le diamètre (§2.2). `ROTATION` donne un exemple de conflit d'intérêts ordinal pur — les meilleures cases des deux joueurs diffèrent. `MUR_PARTIEL` n'est **pas** une chambre : la table de Ligne porte un ex æquo (deux cases au rang 2), c'est un point de l'univers étendu, une face du mur. La question de l'opération 13 est exactement : que se passe-t-il quand un chemin **traverse** de telles faces ?

Un mot sur ce que « jeu ordinal 3×2 » veut dire ici, pour éviter tout malentendu de modélisation. Chaque joueur range les six cases de la matrice 3 rangées × 2 colonnes par ordre de préférence, sans intensité cardinale : seuls les rangs comptent, exactement comme dans le 2×2 de Robinson-Goforth porté par GT-20/GT-3b. Les 720 tables strictes par côté sont les 6! ordres totaux possibles ; une table avec ex æquo est un ordre *partiel* au sens où plusieurs cases partagent un rang. Aucune théorie des équilibres n'est mobilisée dans ce notebook : l'objet d'étude est la **géométrie de l'espace des préférences** — la question de l'opération 13 est une question de métrique sur cet espace, pas de comportement stratégique.

In [5]:
// === Section 2.1 : les generateurs -- swap de facettes, fusion (descente de mur), scission (remontee) ===

use std::collections::BTreeSet;

fn canonique(t: &[u8]) -> Vec<u8> {
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    t.iter().map(|v| vals.iter().position(|w| w == v).unwrap() as u8 + 1).collect()
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

fn tup(t: &[u8]) -> String {
    // Repr Python d'un tuple : (1, 2, 3, 4, 5, 6)
    format!("({})", t.iter().map(|v| v.to_string()).collect::<Vec<_>>().join(", "))
}

fn swap_adjacent(t: &[u8], k: u8) -> Vec<u8> {
    // Traversee de facette k<->k+1 : echange les cases portant les valeurs k et k+1 (tables strictes).
    let pk = t.iter().position(|v| *v == k).unwrap();
    let pk1 = t.iter().position(|v| *v == k + 1).unwrap();
    let mut l = t.to_vec();
    l.swap(pk, pk1);
    l
}

fn fusion(t: &[u8], k: u8) -> Vec<u8> {
    // Descente de mur (le faire_le_tie de GT-20) : coalesce les valeurs k et k+1, forme canonique.
    canonique(&t.iter().map(|v| if *v == k + 1 { k } else { *v }).collect::<Vec<u8>>())
}

fn next_comb(c: &mut Vec<usize>, n: usize) -> bool {
    // Prochaine combinaison croissante de taille c.len() dans 0..n, en ordre lexicographique
    // (= itertools.combinations sur une liste croissante).
    let j = c.len();
    let mut i = j;
    while i > 0 {
        i -= 1;
        if c[i] < n - j + i {
            c[i] += 1;
            for k in i + 1..j {
                c[k] = c[k - 1] + 1;
            }
            return true;
        }
    }
    false
}

fn scissions(t: &[u8]) -> Vec<Vec<u8>> {
    // Remontee de mur : chaque bipartition non triviale d'un bloc d'ex aequo, dans les deux ordres.
    // Ordre impose : v croissant parmi les valeurs distinctes triees, j de 1 a m-1,
    // combinaisons des positions croissantes prises j a la fois en ordre lexicographique.
    let mut res: Vec<Vec<u8>> = vec![];
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    for v in vals {
        let p: Vec<usize> = t.iter().enumerate().filter(|(_, x)| **x == v).map(|(i, _)| i).collect();
        let m = p.len();
        if m < 2 {
            continue;
        }
        for j in 1..m {
            let mut c: Vec<usize> = (0..j).collect();
            loop {
                let s: BTreeSet<usize> = c.iter().map(|idx| p[*idx]).collect();
                let mut u = vec![0u8; t.len()];
                for (i, x) in t.iter().enumerate() {
                    u[i] = if *x == v {
                        if s.contains(&i) { v } else { v + 1 }
                    } else if *x > v {
                        x + 1
                    } else {
                        *x
                    };
                }
                res.push(canonique(&u));
                if !next_comb(&mut c, m) {
                    break;
                }
            }
        }
    }
    res
}

let t: Vec<u8> = vec![1, 2, 3, 4, 5, 6];
println!("Table identite                     : {}", tup(&t));
println!("swap facette 2<->3                 : {}", tup(&swap_adjacent(&t, 2)));
println!("fusion 2~3 (descente de mur)       : {}", tup(&fusion(&t, 2)));
let sc: Vec<String> = scissions(&vec![1, 2, 2, 3, 4, 5]).iter().map(|x| tup(x)).collect();
println!("scissions de la table liee (1,2,2,3,4,5) : [{}]", sc.join(", "));
println!("Le swap de singletons = fusion puis scission dans l'autre sens : {}",
    py_bool(scissions(&fusion(&t, 2)).iter().any(|s| *s == swap_adjacent(&t, 2))));


Table identite                     : (1, 2, 3, 4, 5, 6)


swap facette 2<->3                 : (1, 3, 2, 4, 5, 6)


fusion 2~3 (descente de mur)       : (1, 2, 2, 3, 4, 5)


scissions de la table liee (1,2,2,3,4,5) : [(1, 2, 3, 4, 5, 6), (1, 3, 2, 4, 5, 6)]


Le swap de singletons = fusion puis scission dans l'autre sens : True


### Lecture du résultat — trois familles de générateurs, et le swap décomposé en fusion puis scission

Trois familles de générateurs, et une identité structurelle qui éclaire tout le notebook : **un swap de facettes entre deux valeurs strictes est exactement une fusion suivie d'une scission dans l'ordre opposé** (vérifié ci-dessus). Autrement dit, le permutoèdre des chambres est *plongé* dans le graphe des ordres faibles : chaque arête de chambre se décompose en deux pas de mur. C'est cette décomposition qui rend la question des raccourcis sensée — traverser les murs coûte des pas *supplémentaires* pour réordonner, mais permet de déplacer des **blocs entiers** en une seule scission. Le verdict 2×2 de GT-20 disait que ce troc n'était jamais rentable à 4 cases ; le §4 mesure s'il le reste à 6.

**Les trois modèles de graphe utilisés dans la suite** (déclarés ici, utilisés ensuite) :

| Modèle | Sommets | Arêtes | Question |
|---|---|---|---|
| **Chambres** (permutoèdre) | ordres stricts | swaps de facettes | distance de référence `d_perm` |
| **Murs dirigés** (GT-20 historique) | ordres faibles | swaps + fusions *à sens unique* | reproduction du cadre original |
| **Treillis fermé** (canonique) | ordres faibles | fusions + scissions, non dirigé | un mur se traverse **dans les deux sens** |

Le modèle fermé est celui du graphe de couverture du treillis des ordres faibles — l'objet standard de la combinatoire des arrangements de type A. Il est le seul où « traverser un mur » a un sens honnête : descendre une facette *et pouvoir remonter ailleurs*. C'est lui qui porte la question falsifiable.

Pourquoi ce choix canonique plutôt qu'un autre ? Une modélisation naïve du modèle fermé consisterait à garder les swaps *aussi* sur les tables liées — mais un swap entre deux valeurs liées n'a pas de définition naturelle (échanger quelles occurrences ?), et l'échange de blocs entiers déplace plusieurs cases pour un pas, ce qui fabrique des raccourcis par fiat de modélisation plutôt que par la structure. En restant sur les seules relations de couverture du treillis — fusionner deux blocs adjacents, ou scinder un bloc — chaque pas a exactement une signification : on descend dans la stratification (on renonce à départager deux cases) ou on remonte (on les départage autrement). La question « les murs raccourcissent-ils ? » devient alors une question sur l'objet mathématique standard, pas sur les détails d'une implémentation.

In [6]:
// === Section 2.2 : construction des adjacences, symetrie, connectivite, diametres mesures ===

use std::collections::{BTreeSet, HashMap, VecDeque};

fn canonique(t: &[u8]) -> Vec<u8> {
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    t.iter().map(|v| vals.iter().position(|w| w == v).unwrap() as u8 + 1).collect()
}

fn ordres_faibles(n: u8) -> Vec<Vec<u8>> {
    let mut set = BTreeSet::new();
    let mut t = vec![1u8; n as usize];
    let total = (n as u64).pow(n as u32);
    for _ in 0..total {
        set.insert(canonique(&t));
        let mut i = n as usize;
        while i > 0 {
            i -= 1;
            if t[i] < n {
                t[i] += 1;
                break;
            }
            t[i] = 1;
        }
    }
    set.into_iter().collect()
}

fn nb_distinctes(t: &[u8]) -> usize {
    let mut s = t.to_vec();
    s.sort();
    s.dedup();
    s.len()
}

fn swap_adjacent(t: &[u8], k: u8) -> Vec<u8> {
    let pk = t.iter().position(|v| *v == k).unwrap();
    let pk1 = t.iter().position(|v| *v == k + 1).unwrap();
    let mut l = t.to_vec();
    l.swap(pk, pk1);
    l
}

fn fusion(t: &[u8], k: u8) -> Vec<u8> {
    canonique(&t.iter().map(|v| if *v == k + 1 { k } else { *v }).collect::<Vec<u8>>())
}

fn next_comb(c: &mut Vec<usize>, n: usize) -> bool {
    let j = c.len();
    let mut i = j;
    while i > 0 {
        i -= 1;
        if c[i] < n - j + i {
            c[i] += 1;
            for k in i + 1..j {
                c[k] = c[k - 1] + 1;
            }
            return true;
        }
    }
    false
}

fn scissions(t: &[u8]) -> Vec<Vec<u8>> {
    let mut res: Vec<Vec<u8>> = vec![];
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    for v in vals {
        let p: Vec<usize> = t.iter().enumerate().filter(|(_, x)| **x == v).map(|(i, _)| i).collect();
        let m = p.len();
        if m < 2 {
            continue;
        }
        for j in 1..m {
            let mut c: Vec<usize> = (0..j).collect();
            loop {
                let s: BTreeSet<usize> = c.iter().map(|idx| p[*idx]).collect();
                let mut u = vec![0u8; t.len()];
                for (i, x) in t.iter().enumerate() {
                    u[i] = if *x == v {
                        if s.contains(&i) { v } else { v + 1 }
                    } else if *x > v {
                        x + 1
                    } else {
                        *x
                    };
                }
                res.push(canonique(&u));
                if !next_comb(&mut c, m) {
                    break;
                }
            }
        }
    }
    res
}

fn voisins_permutoedre(t: &[u8], n: u8) -> Vec<Vec<u8>> {
    // k = 1..n-1 croissant.
    (1..n).map(|k| swap_adjacent(t, k)).collect()
}

fn voisins_treillis(t: &[u8], n: u8) -> Vec<Vec<u8>> {
    // Fusions (k et k+1 valeurs presentes) PUIS scissions -- le graphe de couverture du treillis.
    let mut res: Vec<Vec<u8>> = vec![];
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    for k in 1..n {
        if vals.contains(&k) && vals.contains(&(k + 1)) {
            res.push(fusion(t, k));
        }
    }
    res.extend(scissions(t));
    res
}

fn bfs(src: &Vec<u8>, adj: &HashMap<Vec<u8>, Vec<Vec<u8>>>) -> HashMap<Vec<u8>, u32> {
    // BFS exhaustive ; adj est un dictionnaire sommet -> voisinage (le constructeur l'utilisera aussi).
    let mut dist: HashMap<Vec<u8>, u32> = HashMap::new();
    dist.insert(src.clone(), 0);
    let mut q: VecDeque<Vec<u8>> = VecDeque::new();
    q.push_back(src.clone());
    while let Some(u) = q.pop_front() {
        let du = dist[&u];
        for v in &adj[&u] {
            if !dist.contains_key(v) {
                dist.insert(v.clone(), du + 1);
                q.push_back(v.clone());
            }
        }
    }
    dist
}

for n in [4u8, 6u8] {
    let of = ordres_faibles(n);
    let st: Vec<Vec<u8>> = of.iter().filter(|t| nb_distinctes(t) == n as usize).cloned().collect();
    let adj_p: HashMap<Vec<u8>, Vec<Vec<u8>>> = st.iter().map(|t| (t.clone(), voisins_permutoedre(t, n))).collect();
    let adj_c: HashMap<Vec<u8>, Vec<Vec<u8>>> = of.iter().map(|t| (t.clone(), voisins_treillis(t, n))).collect();
    let asym: usize = adj_c.iter().map(|(u, vs)| vs.iter().filter(|v| !adj_c[*v].contains(u)).count()).sum();
    let deg_moy = adj_c.values().map(|v| v.len()).sum::<usize>() as f64 / adj_c.len() as f64;
    let dp = bfs(&st[0], &adj_p);
    let dc = bfs(&of[0], &adj_c);
    let ecc = *dc.values().max().unwrap();
    println!("n={} | stricts={} faibles={} | treillis : asymetries={}, degre moyen={:.1}, connexe={}/{}, excentricite du centre tout-lie={} -> diametre = {}",
        n, st.len(), of.len(), asym, deg_moy, dc.len(), of.len(), ecc, 2 * ecc);
    let dpmax = *dp.values().max().unwrap();
    println!("     | permutoedre : connexe={}/{}, diametre={} (= inversions de l'element le plus long w0 : {})",
        dp.len(), st.len(), dpmax, n as usize * (n as usize - 1) / 2);
};


n=4 | stricts=24 faibles=75 | treillis : asymetries=0, degre moyen=4.2, connexe=75/75, excentricite du centre tout-lie=3 -> diametre = 6


     | permutoedre : connexe=24/24, diametre=6 (= inversions de l'element le plus long w0 : 6)


n=6 | stricts=720 faibles=4683 | treillis : asymetries=0, degre moyen=7.1, connexe=4683/4683, excentricite du centre tout-lie=5 -> diametre = 10


     | permutoedre : connexe=720/720, diametre=15 (= inversions de l'element le plus long w0 : 15)


### Lecture du résultat — les diamètres mesurés : 6 et 15 sur le permutoèdre, 6 et 10 sur le treillis

Deux contrôles structurels passent : le graphe du treillis est **exactement symétrique** (0 asymétrie — la remontée de mur est bien l'inverse de la descente), et les diamètres mesurés du permutoèdre valent 6 (n=4) et 15 (n=6), soit les nombres d'inversions de l'élément le plus long w0 de S4 et S6 — la théorie des groupes de Coxeter prédit ces valeurs et la mesure les confirme.

Pour le treillis, la mesure directe donne l'**excentricité de la table tout-liée** : 3 (n=4) et 5 (n=6). Ce n'est pas le diamètre, mais presque : la table tout-liée est le **centre** du treillis — atteindre une table à r blocs depuis elle demande exactement r−1 scissions, une par bloc créé — donc toute paire de sommets est à au plus 2×(n−1) l'un de l'autre en passant par le centre. Le diamètre du treillis vaut ainsi 6 (n=4) et 10 (n=6), et la borne est *serrée* : le §6 la réalise entre les deux antipodes stricts, certifié par le vérificateur. Premier fait saillant déjà visible à n=6 : 10 < 15 — la contraction métrique existe dans l'univers étendu ; reste à savoir si elle touche les paires de *chambres* (le §4 y répond). À n=4 en revanche, 6 = 6 : premier indice que le 2×2 n'a pas de marge.

In [7]:
// === Section 3 : reproduction du verdict 2x2 de GT-20 sous le modele ferme ===
// GT-20 avait mesure 0 raccourci sur les 576 paires de chambres 2x2, mais avec des murs a sens
// unique (swaps + faire_le_tie diriges) : une fois un mur descendu, impossible de remonter, donc
// le 0 etait en partie structurel. On re-meure ici sous le modele ferme (fusions + scissions).

use std::collections::{BTreeSet, HashMap, VecDeque};

fn canonique(t: &[u8]) -> Vec<u8> {
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    t.iter().map(|v| vals.iter().position(|w| w == v).unwrap() as u8 + 1).collect()
}

fn ordres_faibles(n: u8) -> Vec<Vec<u8>> {
    let mut set = BTreeSet::new();
    let mut t = vec![1u8; n as usize];
    let total = (n as u64).pow(n as u32);
    for _ in 0..total {
        set.insert(canonique(&t));
        let mut i = n as usize;
        while i > 0 {
            i -= 1;
            if t[i] < n {
                t[i] += 1;
                break;
            }
            t[i] = 1;
        }
    }
    set.into_iter().collect()
}

fn nb_distinctes(t: &[u8]) -> usize {
    let mut s = t.to_vec();
    s.sort();
    s.dedup();
    s.len()
}

fn tup(t: &[u8]) -> String {
    format!("({})", t.iter().map(|v| v.to_string()).collect::<Vec<_>>().join(", "))
}

fn swap_adjacent(t: &[u8], k: u8) -> Vec<u8> {
    let pk = t.iter().position(|v| *v == k).unwrap();
    let pk1 = t.iter().position(|v| *v == k + 1).unwrap();
    let mut l = t.to_vec();
    l.swap(pk, pk1);
    l
}

fn fusion(t: &[u8], k: u8) -> Vec<u8> {
    canonique(&t.iter().map(|v| if *v == k + 1 { k } else { *v }).collect::<Vec<u8>>())
}

fn next_comb(c: &mut Vec<usize>, n: usize) -> bool {
    let j = c.len();
    let mut i = j;
    while i > 0 {
        i -= 1;
        if c[i] < n - j + i {
            c[i] += 1;
            for k in i + 1..j {
                c[k] = c[k - 1] + 1;
            }
            return true;
        }
    }
    false
}

fn scissions(t: &[u8]) -> Vec<Vec<u8>> {
    let mut res: Vec<Vec<u8>> = vec![];
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    for v in vals {
        let p: Vec<usize> = t.iter().enumerate().filter(|(_, x)| **x == v).map(|(i, _)| i).collect();
        let m = p.len();
        if m < 2 {
            continue;
        }
        for j in 1..m {
            let mut c: Vec<usize> = (0..j).collect();
            loop {
                let s: BTreeSet<usize> = c.iter().map(|idx| p[*idx]).collect();
                let mut u = vec![0u8; t.len()];
                for (i, x) in t.iter().enumerate() {
                    u[i] = if *x == v {
                        if s.contains(&i) { v } else { v + 1 }
                    } else if *x > v {
                        x + 1
                    } else {
                        *x
                    };
                }
                res.push(canonique(&u));
                if !next_comb(&mut c, m) {
                    break;
                }
            }
        }
    }
    res
}

fn voisins_permutoedre(t: &[u8], n: u8) -> Vec<Vec<u8>> {
    (1..n).map(|k| swap_adjacent(t, k)).collect()
}

fn voisins_treillis(t: &[u8], n: u8) -> Vec<Vec<u8>> {
    let mut res: Vec<Vec<u8>> = vec![];
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    for k in 1..n {
        if vals.contains(&k) && vals.contains(&(k + 1)) {
            res.push(fusion(t, k));
        }
    }
    res.extend(scissions(t));
    res
}

fn bfs(src: &Vec<u8>, adj: &HashMap<Vec<u8>, Vec<Vec<u8>>>) -> HashMap<Vec<u8>, u32> {
    let mut dist: HashMap<Vec<u8>, u32> = HashMap::new();
    dist.insert(src.clone(), 0);
    let mut q: VecDeque<Vec<u8>> = VecDeque::new();
    q.push_back(src.clone());
    while let Some(u) = q.pop_front() {
        let du = dist[&u];
        for v in &adj[&u] {
            if !dist.contains_key(v) {
                dist.insert(v.clone(), du + 1);
                q.push_back(v.clone());
            }
        }
    }
    dist
}

fn recensement(st: &[Vec<u8>], adj_p: &HashMap<Vec<u8>, Vec<Vec<u8>>>, adj_c: &HashMap<Vec<u8>, Vec<Vec<u8>>>) -> (usize, usize, usize, Option<(Vec<u8>, Vec<u8>, u32, u32)>) {
    // Pour chaque paire ordonnee de chambres : d_treillis < d_permutoedre ?
    let mut raccourcis = 0;
    let mut egal = 0;
    let mut paires = 0;
    let mut exemple: Option<(Vec<u8>, Vec<u8>, u32, u32)> = None;
    for a in st {
        let dperm = bfs(a, adj_p);
        let dtl = bfs(a, adj_c);
        for b in st {
            paires += 1;
            if dtl[b] < dperm[b] {
                raccourcis += 1;
                if exemple.is_none() && b > a {
                    exemple = Some((a.clone(), b.clone(), dperm[b], dtl[b]));
                }
            } else if dtl[b] == dperm[b] {
                egal += 1;
            }
        }
    }
    (paires, raccourcis, egal, exemple)
}

fn fmt_exemple(e: &Option<(Vec<u8>, Vec<u8>, u32, u32)>) -> String {
    match e {
        None => "None".to_string(),
        Some((a, b, dp, dt)) => format!("({}, {}, {}, {})", tup(a), tup(b), dp, dt),
    }
}

let of4 = ordres_faibles(4);
let st4: Vec<Vec<u8>> = of4.iter().filter(|t| nb_distinctes(t) == 4).cloned().collect();
let adj_p4: HashMap<Vec<u8>, Vec<Vec<u8>>> = st4.iter().map(|t| (t.clone(), voisins_permutoedre(t, 4))).collect();
let adj_c4: HashMap<Vec<u8>, Vec<Vec<u8>>> = of4.iter().map(|t| (t.clone(), voisins_treillis(t, 4))).collect();

let t0 = std::time::Instant::now();
let (paires4, rc4, eg4, ex4) = recensement(&st4, &adj_p4, &adj_c4);
println!("Recensement 2x2 (modele ferme) : {} paires ordonnees", paires4);
println!("  raccourcis (d_treillis < d_perm) : {}", rc4);
println!("  egalites (aucun gain a traverser) : {}", eg4);
println!("  exemple de raccourci : {}", fmt_exemple(&ex4));
println!("  ({:.1}s)", t0.elapsed().as_secs_f64());


Recensement 2x2 (modele ferme) : 576 paires ordonnees


  raccourcis (d_treillis < d_perm) : 0


  egalites (aucun gain a traverser) : 48


  exemple de raccourci : None


  (0.0s)


### Lecture du résultat — le verdict 2×2 survit au modèle fermé : 0 raccourci sur 576 paires

**Le verdict de GT-20 survit au modèle fermé** : 0 raccourci sur 576 paires, même quand les murs se traversent dans les deux sens. La mesure originale n'était donc pas un artefact du sens unique — à 4 cases, le troc « des pas supplémentaires pour réordonner » n'est jamais rentable, et les 48 égalités (dont l'antipode, 6 = 2×(4−1)) montrent que le meilleur chemin par les murs égale tout au plus le chemin de chambres. La généralisation à tester en §4 est ainsi proprement posée : *le même banc, le même modèle fermé, un seul changement — le substrat.*

In [8]:
// === Section 4 : le recensement exhaustif 3x2 -- la question falsifiable, la reponse mesuree ===

use std::collections::{BTreeSet, HashMap, VecDeque};

fn canonique(t: &[u8]) -> Vec<u8> {
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    t.iter().map(|v| vals.iter().position(|w| w == v).unwrap() as u8 + 1).collect()
}

fn ordres_faibles(n: u8) -> Vec<Vec<u8>> {
    let mut set = BTreeSet::new();
    let mut t = vec![1u8; n as usize];
    let total = (n as u64).pow(n as u32);
    for _ in 0..total {
        set.insert(canonique(&t));
        let mut i = n as usize;
        while i > 0 {
            i -= 1;
            if t[i] < n {
                t[i] += 1;
                break;
            }
            t[i] = 1;
        }
    }
    set.into_iter().collect()
}

fn nb_distinctes(t: &[u8]) -> usize {
    let mut s = t.to_vec();
    s.sort();
    s.dedup();
    s.len()
}

fn tup(t: &[u8]) -> String {
    format!("({})", t.iter().map(|v| v.to_string()).collect::<Vec<_>>().join(", "))
}

fn swap_adjacent(t: &[u8], k: u8) -> Vec<u8> {
    let pk = t.iter().position(|v| *v == k).unwrap();
    let pk1 = t.iter().position(|v| *v == k + 1).unwrap();
    let mut l = t.to_vec();
    l.swap(pk, pk1);
    l
}

fn fusion(t: &[u8], k: u8) -> Vec<u8> {
    canonique(&t.iter().map(|v| if *v == k + 1 { k } else { *v }).collect::<Vec<u8>>())
}

fn next_comb(c: &mut Vec<usize>, n: usize) -> bool {
    let j = c.len();
    let mut i = j;
    while i > 0 {
        i -= 1;
        if c[i] < n - j + i {
            c[i] += 1;
            for k in i + 1..j {
                c[k] = c[k - 1] + 1;
            }
            return true;
        }
    }
    false
}

fn scissions(t: &[u8]) -> Vec<Vec<u8>> {
    let mut res: Vec<Vec<u8>> = vec![];
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    for v in vals {
        let p: Vec<usize> = t.iter().enumerate().filter(|(_, x)| **x == v).map(|(i, _)| i).collect();
        let m = p.len();
        if m < 2 {
            continue;
        }
        for j in 1..m {
            let mut c: Vec<usize> = (0..j).collect();
            loop {
                let s: BTreeSet<usize> = c.iter().map(|idx| p[*idx]).collect();
                let mut u = vec![0u8; t.len()];
                for (i, x) in t.iter().enumerate() {
                    u[i] = if *x == v {
                        if s.contains(&i) { v } else { v + 1 }
                    } else if *x > v {
                        x + 1
                    } else {
                        *x
                    };
                }
                res.push(canonique(&u));
                if !next_comb(&mut c, m) {
                    break;
                }
            }
        }
    }
    res
}

fn voisins_permutoedre(t: &[u8], n: u8) -> Vec<Vec<u8>> {
    (1..n).map(|k| swap_adjacent(t, k)).collect()
}

fn voisins_treillis(t: &[u8], n: u8) -> Vec<Vec<u8>> {
    let mut res: Vec<Vec<u8>> = vec![];
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    for k in 1..n {
        if vals.contains(&k) && vals.contains(&(k + 1)) {
            res.push(fusion(t, k));
        }
    }
    res.extend(scissions(t));
    res
}

fn bfs(src: &Vec<u8>, adj: &HashMap<Vec<u8>, Vec<Vec<u8>>>) -> HashMap<Vec<u8>, u32> {
    let mut dist: HashMap<Vec<u8>, u32> = HashMap::new();
    dist.insert(src.clone(), 0);
    let mut q: VecDeque<Vec<u8>> = VecDeque::new();
    q.push_back(src.clone());
    while let Some(u) = q.pop_front() {
        let du = dist[&u];
        for v in &adj[&u] {
            if !dist.contains_key(v) {
                dist.insert(v.clone(), du + 1);
                q.push_back(v.clone());
            }
        }
    }
    dist
}

fn recensement(st: &[Vec<u8>], adj_p: &HashMap<Vec<u8>, Vec<Vec<u8>>>, adj_c: &HashMap<Vec<u8>, Vec<Vec<u8>>>) -> (usize, usize, usize, Option<(Vec<u8>, Vec<u8>, u32, u32)>) {
    let mut raccourcis = 0;
    let mut egal = 0;
    let mut paires = 0;
    let mut exemple: Option<(Vec<u8>, Vec<u8>, u32, u32)> = None;
    for a in st {
        let dperm = bfs(a, adj_p);
        let dtl = bfs(a, adj_c);
        for b in st {
            paires += 1;
            if dtl[b] < dperm[b] {
                raccourcis += 1;
                if exemple.is_none() && b > a {
                    exemple = Some((a.clone(), b.clone(), dperm[b], dtl[b]));
                }
            } else if dtl[b] == dperm[b] {
                egal += 1;
            }
        }
    }
    (paires, raccourcis, egal, exemple)
}

let of6 = ordres_faibles(6);
let st6: Vec<Vec<u8>> = of6.iter().filter(|t| nb_distinctes(t) == 6).cloned().collect();
let adj_p6: HashMap<Vec<u8>, Vec<Vec<u8>>> = st6.iter().map(|t| (t.clone(), voisins_permutoedre(t, 6))).collect();
let adj_c6: HashMap<Vec<u8>, Vec<Vec<u8>>> = of6.iter().map(|t| (t.clone(), voisins_treillis(t, 6))).collect();

let t0 = std::time::Instant::now();
let (paires6, rc6, eg6, ex6) = recensement(&st6, &adj_p6, &adj_c6);
let duree = t0.elapsed().as_secs_f64();
println!("Recensement 3x2 (modele ferme) : {} paires ordonnees de chambres (720 x 720)", paires6);
println!("  raccourcis (d_treillis < d_perm) : {}  ({:.1} %)", rc6, 100.0 * rc6 as f64 / paires6 as f64);
println!("  egalites                         : {}", eg6);
let ligne_exemple = match &ex6 {
    None => "  exemple : None".to_string(),
    Some((a, b, dp, dt)) => format!("  exemple : {} -> {} : d_perm = {}, d_treillis = {}", tup(a), tup(b), dp, dt),
};
println!("{}", ligne_exemple);
println!("  diametre permutoedre = 15 | diametre treillis = 10 (centre tout-lie d'excentricite 5, section 2.2)");
println!("  ({:.1}s d'execution -- exhaustive, pas d'echantillonnage)", duree);


Recensement 3x2 (modele ferme) : 518400 paires ordonnees de chambres (720 x 720)


  raccourcis (d_treillis < d_perm) : 77760  (15.0 %)


  egalites                         : 65520


  exemple : (1, 2, 3, 4, 5, 6) -> (1, 5, 6, 4, 3, 2) : d_perm = 9, d_treillis = 8


  diametre permutoedre = 15 | diametre treillis = 10 (centre tout-lie d'excentricite 5, section 2.2)


  (1.8s d'execution -- exhaustive, pas d'echantillonnage)


### Lecture du résultat — la généralisation est réfutée : 77 760 paires sur 518 400, diamètre contracté de 15 à 10

**La généralisation est réfutée.** Sur le substrat 3×2, 77 760 paires ordonnées de chambres sur 518 400 — **15,0 %** — ont un chemin plus court en traversant les murs qu'en restant dans les chambres, et le diamètre par côté se contracte de 15 à 10. Le verdict 2×2 (« la stratification ne comprime pas la métrique ») n'était pas un théorème de l'opération : c'était une propriété du petit substrat.

**Pourquoi 10, et pourquoi par là.** Le chemin intégral par les murs — fusionner tout (5 fusions successives jusqu'à la table tout-liée), puis scissionner dans l'ordre cible (5 remontées) — coûte exactement 2×(6−1) = 10 : c'est lui qui réalise le nouveau diamètre entre antipodes, là où le permutoèdre en demande 15. L'économie vient des **blocs** : une seule scission d'un bloc de taille m replace m cases d'un coup, quand le permutoèdre doit les permuter une à une.

**L'arithmétique du troc** mérite d'être écrite noir sur blanc, car c'est elle qui explique la frontière 2×2 / 3×2. Toute traversée utile de mur coûte *au moins* 2 pas (une descente, une remontée) — pour rentabiliser, le déplacement de blocs réalisé entre les deux doit économiser au moins 2 swaps. Or échanger les positions de deux éléments séparés par j rangs coûte **2j - 1** swaps au permutoèdre (chacun des deux éléments passe les j - 1 éléments qui les séparent, plus l'autre : 2(j - 1) + 1), tandis qu'une scission tranche d'un geste toutes les ex æquo entre les deux sections qu'elle sépare. Sur l'antipode, la loi est lisible et mesurable : renverser les n cases coûte C(n, 2) swaps au permutoèdre (6 à n = 4, 15 à n = 6 — §2.2) et 2(n - 1) pas par les murs (6 — §3, et 10 — §6), d'où un gain net de C(n, 2) - 2(n - 1) = (n - 1)(n - 4)/2 : **0 à 4 cases**, **5 à 6 cases**. À 4 cases le troc ne peut donc rien gagner — le recensement du §3 le mesure : jamais. À 6 cases, la table tout-liée offre un bloc de taille 6 : y loger l'ordre renversé ne demande que 5 scissions (chacune détache une case du bloc), là où le permutoèdre exige un swap par paire inversée — d'où le gain net de 5, exactement mesuré au §4. La question de savoir si *toutes* les paires gagnent (non : 85 % des paires ne raccourcissent pas — et parmi elles, la majorité des trajets sont même plus longs par les murs) et de combien est précisément ce que le recensement mesure, et l'exercice 2 y revient par le bout des antipodes.

C'est exactement le type de résultat que la table des opérations [#12204](https://github.com/jsboige/CoursIA/issues/12204) veut pour l'opération 13 : non pas « on rejoue le témoin », mais **le témoin produit une connaissance nouvelle sur la frontière de la loi** — ici, la borne où « traverser un mur » cesse d'être métriquement neutre et devient un raccourci structurel.

In [9]:
// === Section 5 : le produit de jeux et son lemme de decomposition ===
// Les generateurs ne touchent qu'un cote a la fois : le graphe de jeux est le produit cartesien
// des deux graphes par cote. Lemme : d_jeu(G, H) = d_cote(row_G, row_H) + d_cote(col_G, col_H),
// dans CHACUN des deux modeles (chambres et treillis). Verification directe sur l'antipode.

use std::collections::{HashMap, VecDeque};

fn swap_adjacent(t: &[u8], k: u8) -> Vec<u8> {
    let pk = t.iter().position(|v| *v == k).unwrap();
    let pk1 = t.iter().position(|v| *v == k + 1).unwrap();
    let mut l = t.to_vec();
    l.swap(pk, pk1);
    l
}

fn next_permutation(p: &mut [u8]) -> bool {
    // Prochaine permutation en ordre lexicographique (= l'ordre strict du BTreeSet de la section 1.1).
    let n = p.len();
    let mut i = n;
    loop {
        if i < 2 {
            return false;
        }
        i -= 1;
        if p[i - 1] < p[i] {
            break;
        }
    }
    let mut j = n;
    loop {
        j -= 1;
        if p[j] > p[i - 1] {
            break;
        }
    }
    p.swap(i - 1, j);
    p[i..].reverse();
    true
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

// Les 720 tables strictes en ordre lexicographique, indexees ; jeu = (index_ligne, index_colonne).
let mut tables: Vec<Vec<u8>> = Vec::new();
let mut cur = vec![1u8, 2, 3, 4, 5, 6];
loop {
    tables.push(cur.clone());
    if !next_permutation(&mut cur) {
        break;
    }
}
let mut index: HashMap<Vec<u8>, usize> = HashMap::new();
for (i, t) in tables.iter().enumerate() {
    index.insert(t.clone(), i);
}
let nb = tables.len();
let id_row = index[&vec![1u8, 2, 3, 4, 5, 6]];
let rev_row = index[&vec![6u8, 5, 4, 3, 2, 1]];

let t0 = std::time::Instant::now();
let mut dist = vec![u32::MAX; nb * nb];
let start = id_row * nb + id_row;
dist[start] = 0;
let mut q: VecDeque<usize> = VecDeque::new();
q.push_back(start);
while let Some(s) = q.pop_front() {
    let (a, b) = (s / nb, s % nb);
    let ds = dist[s];
    for k in 1u8..6 {
        let v = swap_adjacent(&tables[a], k);
        let j = index[&v];
        let t = j * nb + b;
        if dist[t] == u32::MAX {
            dist[t] = ds + 1;
            q.push_back(t);
        }
    }
    for k in 1u8..6 {
        let v = swap_adjacent(&tables[b], k);
        let j = index[&v];
        let t = a * nb + j;
        if dist[t] == u32::MAX {
            dist[t] = ds + 1;
            q.push_back(t);
        }
    }
}
let atteint = dist.iter().filter(|d| **d != u32::MAX).count();
let d_directe = dist[rev_row * nb + rev_row];
let d_decomposee = 15 + 15;
println!("BFS directe sur le produit des 518 400 chambres : {} jeux atteints depuis Identite", atteint);
println!("distance Identite -> Renversement : directe = {} | par decomposition = {}", d_directe, d_decomposee);
println!("concordance : {}  ({:.1}s)", py_bool(d_directe == d_decomposee), t0.elapsed().as_secs_f64());


BFS directe sur le produit des 518 400 chambres : 518400 jeux atteints depuis Identite


distance Identite -> Renversement : directe = 30 | par decomposition = 30


concordance : True  (0.4s)


### Lecture du résultat — le lemme de décomposition contrôlé par BFS : 30 = 15 + 15

La BFS directe sur le produit des chambres (518 400 sommets, degré 10) confirme le lemme de décomposition : 30 = 15 + 15. Ce lemme est ce qui garde le substrat 3×2 **tractable honnêtement** : l'univers complet de jeux avec murs des deux côtés compte 4 683² ≈ 21,9 millions de sommets — on ne l'énumère pas, on le couvre : toute distance de jeu s'obtient en sommant deux distances de côté, dans les deux modèles. Le recensement du §4, exhaustif *par côté*, est donc exhaustif *pour les jeux* au sens du lemme. (La BFS produit ci-dessus n'est pas une preuve du lemme — elle en est un contrôle ; la preuve est l'argument : chaque pas touche un seul côté.)

Pourquoi garder ce contrôle coûteux alors que l'argument est simple ? Parce que le lemme porte tout le notebook : si l'entrelacement des générateurs cachait une subtilité — par exemple un swap simultané des deux côtés compté pour un pas, ou une interaction entre les murs de Ligne et ceux de Colonne —, toutes les distances de jeu dérivées du §4 seraient fausses *sans aucun signal d'erreur local*. Le contrôle direct sur l'antipode (le cas où l'erreur éventuelle serait maximale, 30 étant le diamètre) est le témoin qui aurait le contradire le premier. C'est la même discipline que le vérificateur du §6 : on ne croit pas un argument élégant sans un recalcul indépendant là où il est le plus vulnérable.

In [10]:
// === Section 6.1 : construire_chemin -- le constructeur temoin (portage GT-20) ===

use std::collections::{BTreeSet, HashMap, VecDeque};

fn canonique(t: &[u8]) -> Vec<u8> {
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    t.iter().map(|v| vals.iter().position(|w| w == v).unwrap() as u8 + 1).collect()
}

fn ordres_faibles(n: u8) -> Vec<Vec<u8>> {
    let mut set = BTreeSet::new();
    let mut t = vec![1u8; n as usize];
    let total = (n as u64).pow(n as u32);
    for _ in 0..total {
        set.insert(canonique(&t));
        let mut i = n as usize;
        while i > 0 {
            i -= 1;
            if t[i] < n {
                t[i] += 1;
                break;
            }
            t[i] = 1;
        }
    }
    set.into_iter().collect()
}

fn nb_distinctes(t: &[u8]) -> usize {
    let mut s = t.to_vec();
    s.sort();
    s.dedup();
    s.len()
}

fn tup(t: &[u8]) -> String {
    format!("({})", t.iter().map(|v| v.to_string()).collect::<Vec<_>>().join(", "))
}

fn swap_adjacent(t: &[u8], k: u8) -> Vec<u8> {
    let pk = t.iter().position(|v| *v == k).unwrap();
    let pk1 = t.iter().position(|v| *v == k + 1).unwrap();
    let mut l = t.to_vec();
    l.swap(pk, pk1);
    l
}

fn fusion(t: &[u8], k: u8) -> Vec<u8> {
    canonique(&t.iter().map(|v| if *v == k + 1 { k } else { *v }).collect::<Vec<u8>>())
}

fn next_comb(c: &mut Vec<usize>, n: usize) -> bool {
    let j = c.len();
    let mut i = j;
    while i > 0 {
        i -= 1;
        if c[i] < n - j + i {
            c[i] += 1;
            for k in i + 1..j {
                c[k] = c[k - 1] + 1;
            }
            return true;
        }
    }
    false
}

fn scissions(t: &[u8]) -> Vec<Vec<u8>> {
    let mut res: Vec<Vec<u8>> = vec![];
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    for v in vals {
        let p: Vec<usize> = t.iter().enumerate().filter(|(_, x)| **x == v).map(|(i, _)| i).collect();
        let m = p.len();
        if m < 2 {
            continue;
        }
        for j in 1..m {
            let mut c: Vec<usize> = (0..j).collect();
            loop {
                let s: BTreeSet<usize> = c.iter().map(|idx| p[*idx]).collect();
                let mut u = vec![0u8; t.len()];
                for (i, x) in t.iter().enumerate() {
                    u[i] = if *x == v {
                        if s.contains(&i) { v } else { v + 1 }
                    } else if *x > v {
                        x + 1
                    } else {
                        *x
                    };
                }
                res.push(canonique(&u));
                if !next_comb(&mut c, m) {
                    break;
                }
            }
        }
    }
    res
}

fn voisins_permutoedre(t: &[u8], n: u8) -> Vec<Vec<u8>> {
    (1..n).map(|k| swap_adjacent(t, k)).collect()
}

fn voisins_treillis(t: &[u8], n: u8) -> Vec<Vec<u8>> {
    let mut res: Vec<Vec<u8>> = vec![];
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    for k in 1..n {
        if vals.contains(&k) && vals.contains(&(k + 1)) {
            res.push(fusion(t, k));
        }
    }
    res.extend(scissions(t));
    res
}

fn bfs(src: &Vec<u8>, adj: &HashMap<Vec<u8>, Vec<Vec<u8>>>) -> HashMap<Vec<u8>, u32> {
    let mut dist: HashMap<Vec<u8>, u32> = HashMap::new();
    dist.insert(src.clone(), 0);
    let mut q: VecDeque<Vec<u8>> = VecDeque::new();
    q.push_back(src.clone());
    while let Some(u) = q.pop_front() {
        let du = dist[&u];
        for v in &adj[&u] {
            if !dist.contains_key(v) {
                dist.insert(v.clone(), du + 1);
                q.push_back(v.clone());
            }
        }
    }
    dist
}

fn construire_chemin(depart: &Vec<u8>, arrivee: &Vec<u8>, adj: &HashMap<Vec<u8>, Vec<Vec<u8>>>) -> Option<Vec<(Vec<u8>, Vec<u8>)>> {
    // Produit une suite (table, table) de depart a arrivee par BFS + remontee des parents.
    // Discipline : pop FIFO, break quand arrivee est DEFILE, parent fixe a la premiere decouverte.
    let mut parent: HashMap<Vec<u8>, Option<Vec<u8>>> = HashMap::new();
    parent.insert(depart.clone(), None);
    let mut q: VecDeque<Vec<u8>> = VecDeque::new();
    q.push_back(depart.clone());
    while let Some(u) = q.pop_front() {
        if u == *arrivee {
            break;
        }
        for v in &adj[&u] {
            if !parent.contains_key(v) {
                parent.insert(v.clone(), Some(u.clone()));
                q.push_back(v.clone());
            }
        }
    }
    if !parent.contains_key(arrivee) {
        return None;
    }
    let mut chaine: Vec<(Vec<u8>, Vec<u8>)> = vec![];
    let mut cur = arrivee.clone();
    while let Some(pp) = parent[&cur].clone() {
        chaine.push((pp.clone(), cur.clone()));
        cur = pp;
    }
    chaine.reverse();
    Some(chaine)
}

let identite: Vec<u8> = vec![1, 2, 3, 4, 5, 6];
let renverse: Vec<u8> = vec![6, 5, 4, 3, 2, 1];
let of6 = ordres_faibles(6);
let st6: Vec<Vec<u8>> = of6.iter().filter(|t| nb_distinctes(t) == 6).cloned().collect();
let adj_p: HashMap<Vec<u8>, Vec<Vec<u8>>> = st6.iter().map(|t| (t.clone(), voisins_permutoedre(t, 6))).collect();
let adj_c: HashMap<Vec<u8>, Vec<Vec<u8>>> = of6.iter().map(|t| (t.clone(), voisins_treillis(t, 6))).collect();

let cp = construire_chemin(&identite, &renverse, &adj_p).unwrap_or_else(Vec::new);
let cm = construire_chemin(&identite, &renverse, &adj_c).unwrap_or_else(Vec::new);
println!("Chemin de chambres Identite -> Renversement (cote Ligne) : {} pas", cp.len());
for i in 0..5.min(cp.len()) {
    println!("  pas {:>2} : {} -> {}", i + 1, tup(&cp[i].0), tup(&cp[i].1));
}
println!("  ...");
println!("Chemin par les murs  Identite -> Renversement (cote Ligne) : {} pas", cm.len());
for (i, (a, b)) in cm.iter().enumerate() {
    println!("  pas {:>2} : {} -> {}", i + 1, tup(a), tup(b));
};


Chemin de chambres Identite -> Renversement (cote Ligne) : 15 pas


  pas  1 : (1, 2, 3, 4, 5, 6) -> (2, 1, 3, 4, 5, 6)


  pas  2 : (2, 1, 3, 4, 5, 6) -> (3, 1, 2, 4, 5, 6)


  pas  3 : (3, 1, 2, 4, 5, 6) -> (3, 2, 1, 4, 5, 6)


  pas  4 : (3, 2, 1, 4, 5, 6) -> (4, 2, 1, 3, 5, 6)


  pas  5 : (4, 2, 1, 3, 5, 6) -> (4, 3, 1, 2, 5, 6)


... (8 outputs left out)

  pas  7 : (2, 2, 2, 2, 2, 1) -> (3, 3, 3, 3, 2, 1)


  pas  8 : (3, 3, 3, 3, 2, 1) -> (4, 4, 4, 3, 2, 1)


  pas  9 : (4, 4, 4, 3, 2, 1) -> (5, 5, 4, 3, 2, 1)


  pas 10 : (5, 5, 4, 3, 2, 1) -> (6, 5, 4, 3, 2, 1)


### Lecture du résultat — le témoin de l'opération 13 : 15 pas par les chambres, 10 par les murs, 5 fusions puis 5 scissions

Le constructeur produit les deux témoins de côté : 15 pas par les chambres, **10 pas par les murs** — et l'affichage du chemin de 10 pas révèle sa structure : les 5 premières étapes sont des **fusions** (la table se lie progressivement jusqu'à la table tout-liée `(1, 1, 1, 1, 1, 1)`), les 5 suivantes des **scissions** qui relâchent les blocs directement dans l'ordre renversé. Le chemin traverse la *cellule centrale* du treillis — le mur total — et c'est précisément cette traversée qui vaut 5 pas d'économie. C'est le témoin de l'opération 13 : un mur réellement traversé, dans les deux sens, produisant un gain mesuré.

In [11]:
// === Section 6.2 : verifier_chemin -- le verificateur independant (Loi II) ===
// Le verificateur ne reutilise NI l'adjacence du constructeur, NI ses structures : il re-ecrit
// ses propres predicats et refait une recherche exhaustive pour la preuve de minimalite.
// Deux modeles, deux verificateurs, chacun re-derive du zero.

use std::collections::{BTreeSet, HashMap, VecDeque};

fn canonique(t: &[u8]) -> Vec<u8> {
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    t.iter().map(|v| vals.iter().position(|w| w == v).unwrap() as u8 + 1).collect()
}

fn ordres_faibles(n: u8) -> Vec<Vec<u8>> {
    let mut set = BTreeSet::new();
    let mut t = vec![1u8; n as usize];
    let total = (n as u64).pow(n as u32);
    for _ in 0..total {
        set.insert(canonique(&t));
        let mut i = n as usize;
        while i > 0 {
            i -= 1;
            if t[i] < n {
                t[i] += 1;
                break;
            }
            t[i] = 1;
        }
    }
    set.into_iter().collect()
}

fn nb_distinctes(t: &[u8]) -> usize {
    let mut s = t.to_vec();
    s.sort();
    s.dedup();
    s.len()
}

fn tup(t: &[u8]) -> String {
    format!("({})", t.iter().map(|v| v.to_string()).collect::<Vec<_>>().join(", "))
}

fn swap_adjacent(t: &[u8], k: u8) -> Vec<u8> {
    let pk = t.iter().position(|v| *v == k).unwrap();
    let pk1 = t.iter().position(|v| *v == k + 1).unwrap();
    let mut l = t.to_vec();
    l.swap(pk, pk1);
    l
}

fn fusion(t: &[u8], k: u8) -> Vec<u8> {
    canonique(&t.iter().map(|v| if *v == k + 1 { k } else { *v }).collect::<Vec<u8>>())
}

fn next_comb(c: &mut Vec<usize>, n: usize) -> bool {
    let j = c.len();
    let mut i = j;
    while i > 0 {
        i -= 1;
        if c[i] < n - j + i {
            c[i] += 1;
            for k in i + 1..j {
                c[k] = c[k - 1] + 1;
            }
            return true;
        }
    }
    false
}

fn scissions(t: &[u8]) -> Vec<Vec<u8>> {
    let mut res: Vec<Vec<u8>> = vec![];
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    for v in vals {
        let p: Vec<usize> = t.iter().enumerate().filter(|(_, x)| **x == v).map(|(i, _)| i).collect();
        let m = p.len();
        if m < 2 {
            continue;
        }
        for j in 1..m {
            let mut c: Vec<usize> = (0..j).collect();
            loop {
                let s: BTreeSet<usize> = c.iter().map(|idx| p[*idx]).collect();
                let mut u = vec![0u8; t.len()];
                for (i, x) in t.iter().enumerate() {
                    u[i] = if *x == v {
                        if s.contains(&i) { v } else { v + 1 }
                    } else if *x > v {
                        x + 1
                    } else {
                        *x
                    };
                }
                res.push(canonique(&u));
                if !next_comb(&mut c, m) {
                    break;
                }
            }
        }
    }
    res
}

fn voisins_permutoedre(t: &[u8], n: u8) -> Vec<Vec<u8>> {
    (1..n).map(|k| swap_adjacent(t, k)).collect()
}

fn voisins_treillis(t: &[u8], n: u8) -> Vec<Vec<u8>> {
    let mut res: Vec<Vec<u8>> = vec![];
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    for k in 1..n {
        if vals.contains(&k) && vals.contains(&(k + 1)) {
            res.push(fusion(t, k));
        }
    }
    res.extend(scissions(t));
    res
}

fn bfs(src: &Vec<u8>, adj: &HashMap<Vec<u8>, Vec<Vec<u8>>>) -> HashMap<Vec<u8>, u32> {
    let mut dist: HashMap<Vec<u8>, u32> = HashMap::new();
    dist.insert(src.clone(), 0);
    let mut q: VecDeque<Vec<u8>> = VecDeque::new();
    q.push_back(src.clone());
    while let Some(u) = q.pop_front() {
        let du = dist[&u];
        for v in &adj[&u] {
            if !dist.contains_key(v) {
                dist.insert(v.clone(), du + 1);
                q.push_back(v.clone());
            }
        }
    }
    dist
}

fn construire_chemin(depart: &Vec<u8>, arrivee: &Vec<u8>, adj: &HashMap<Vec<u8>, Vec<Vec<u8>>>) -> Option<Vec<(Vec<u8>, Vec<u8>)>> {
    let mut parent: HashMap<Vec<u8>, Option<Vec<u8>>> = HashMap::new();
    parent.insert(depart.clone(), None);
    let mut q: VecDeque<Vec<u8>> = VecDeque::new();
    q.push_back(depart.clone());
    while let Some(u) = q.pop_front() {
        if u == *arrivee {
            break;
        }
        for v in &adj[&u] {
            if !parent.contains_key(v) {
                parent.insert(v.clone(), Some(u.clone()));
                q.push_back(v.clone());
            }
        }
    }
    if !parent.contains_key(arrivee) {
        return None;
    }
    let mut chaine: Vec<(Vec<u8>, Vec<u8>)> = vec![];
    let mut cur = arrivee.clone();
    while let Some(pp) = parent[&cur].clone() {
        chaine.push((pp.clone(), cur.clone()));
        cur = pp;
    }
    chaine.reverse();
    Some(chaine)
}

// --- Re-derivation independante (aucune ligne partagee avec la section 2.1 ci-dessus) ---

fn relabel(u: &[u8]) -> Vec<u8> {
    let mut rk = u.to_vec();
    rk.sort();
    rk.dedup();
    u.iter().map(|v| rk.iter().position(|w| w == v).unwrap() as u8 + 1).collect()
}

fn voisins_treillis_verif(t: &[u8]) -> Vec<Vec<u8>> {
    // Fusions : v==k+1 -> k puis relabelage par sorted(set) ; scissions par masques bits
    // (ordre d'enumeration libre : seule l'appartenance est testee).
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    let mut out: Vec<Vec<u8>> = vec![];
    for k in 1u8..6 {
        if vals.contains(&k) && vals.contains(&(k + 1)) {
            let merged: Vec<u8> = t.iter().map(|v| if *v == k + 1 { k } else { *v }).collect();
            out.push(relabel(&merged));
        }
    }
    for v in vals {
        let pos: Vec<usize> = t.iter().enumerate().filter(|(_, x)| **x == v).map(|(i, _)| i).collect();
        if pos.len() < 2 {
            continue;
        }
        for mask in 1..(1usize << pos.len()) - 1 {
            let bas: BTreeSet<usize> = (0..pos.len()).filter(|j| mask >> j & 1 == 1).map(|j| pos[j]).collect();
            let mut nouv = vec![0u8; 6];
            for i in 0..6 {
                nouv[i] = if bas.contains(&i) {
                    v
                } else if t[i] == v {
                    v + 1
                } else {
                    let x = t[i];
                    if x > v { x + 1 } else { x }
                };
            }
            out.push(relabel(&nouv));
        }
    }
    out
}

fn voisins_perm_verif(t: &[u8]) -> Vec<Vec<u8>> {
    // Echange des VALEURS k et k+1 (code distinct de swap_adjacent qui echange des positions).
    let mut out: Vec<Vec<u8>> = vec![];
    for k in 1u8..6 {
        let u: Vec<u8> = t.iter().map(|v| if *v == k { v + 1 } else if *v == k + 1 { v - 1 } else { *v }).collect();
        out.push(u);
    }
    out
}

fn _bfs_depart(src: &Vec<u8>, voisins_fn: fn(&[u8]) -> Vec<Vec<u8>>) -> HashMap<Vec<u8>, u32> {
    let mut dist: HashMap<Vec<u8>, u32> = HashMap::new();
    dist.insert(src.clone(), 0);
    let mut q: VecDeque<Vec<u8>> = VecDeque::new();
    q.push_back(src.clone());
    while let Some(u) = q.pop_front() {
        let du = dist[&u];
        for v in voisins_fn(&u) {
            if !dist.contains_key(&v) {
                dist.insert(v.clone(), du + 1);
                q.push_back(v);
            }
        }
    }
    dist
}

fn verdict_chemin(depart: &Vec<u8>, arrivee: &Vec<u8>, chaine: &[(Vec<u8>, Vec<u8>)], voisins_fn: fn(&[u8]) -> Vec<Vec<u8>>, nom_modele: &str) -> String {
    // Verdict independant : extremites, elementarite de chaque pas, minimalite exhaustive.
    if chaine.is_empty() {
        return "INVALIDE : chemin vide".to_string();
    }
    if chaine[0].0 != *depart || chaine[chaine.len() - 1].1 != *arrivee {
        return "INVALIDE : les extremites ne sont pas celles annoncees".to_string();
    }
    for (pred, cur) in chaine {
        if !voisins_fn(pred).iter().any(|v| *v == *cur) {
            return format!("INVALIDE : pas non elementaire {} -> {}", tup(pred), tup(cur));
        }
    }
    let dist = _bfs_depart(depart, voisins_fn);
    if chaine.len() as u32 != dist[arrivee] {
        return format!("INVALIDE : non minimal ({} pas, distance reelle {})", chaine.len(), dist[arrivee]);
    }
    format!("VALIDE [{}] : {} pas, minimalite certifiee par recherche exhaustive independante", nom_modele, chaine.len())
}

let identite: Vec<u8> = vec![1, 2, 3, 4, 5, 6];
let renverse: Vec<u8> = vec![6, 5, 4, 3, 2, 1];
let of6 = ordres_faibles(6);
let st6: Vec<Vec<u8>> = of6.iter().filter(|t| nb_distinctes(t) == 6).cloned().collect();
let adj_p: HashMap<Vec<u8>, Vec<Vec<u8>>> = st6.iter().map(|t| (t.clone(), voisins_permutoedre(t, 6))).collect();
let adj_c: HashMap<Vec<u8>, Vec<Vec<u8>>> = of6.iter().map(|t| (t.clone(), voisins_treillis(t, 6))).collect();

let chemin_perm = construire_chemin(&identite, &renverse, &adj_p).unwrap_or_else(Vec::new);
let chemin_murs = construire_chemin(&identite, &renverse, &adj_c).unwrap_or_else(Vec::new);

println!("Chemin de chambres (15 pas) juge par le verificateur du permutoedre :");
println!("  {}", verdict_chemin(&identite, &renverse, &chemin_perm, voisins_perm_verif, "permutoedre"));
println!("Chemin par les murs (10 pas) juge par le verificateur du treillis :");
println!("  {}", verdict_chemin(&identite, &renverse, &chemin_murs, voisins_treillis_verif, "treillis"));


Chemin de chambres (15 pas) juge par le verificateur du permutoedre :


  VALIDE [permutoedre] : 15 pas, minimalite certifiee par recherche exhaustive independante


Chemin par les murs (10 pas) juge par le verificateur du treillis :


  VALIDE [treillis] : 10 pas, minimalite certifiee par recherche exhaustive independante


### Lecture du résultat — deux minimalités dans deux graphes, certifiées par deux vérificateurs sans ligne commune

Chaque témoin est certifié **dans son propre modèle**, par un vérificateur qui ré-écrit tout : le chemin de 15 pas est validé élémentaire et minimal au sens des swaps (vérificateur du permutoèdre), le chemin de 10 pas au sens fusions/scissions (vérificateur du treillis) — comparez les deux implémentations de voisinage : elles ne partagent aucune ligne, c'est la séparation de la Loi II. Un point mérite d'être souligné : le chemin de chambres n'est **pas** un chemin du treillis — chacun de ses swaps s'y décompose en fusion + scission (§2.1). Les 15 et les 10 sont donc deux minimalités de deux graphes différents, et c'est précisément leur écart — 5 pas sur l'antipode — qui constitue la mesure de la contraction métrique du §4.

In [12]:
// === Section 7 : controle negatif -- le verificateur doit aussi savoir dire NON ===

use std::collections::{BTreeSet, HashMap, VecDeque};

fn canonique(t: &[u8]) -> Vec<u8> {
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    t.iter().map(|v| vals.iter().position(|w| w == v).unwrap() as u8 + 1).collect()
}

fn ordres_faibles(n: u8) -> Vec<Vec<u8>> {
    let mut set = BTreeSet::new();
    let mut t = vec![1u8; n as usize];
    let total = (n as u64).pow(n as u32);
    for _ in 0..total {
        set.insert(canonique(&t));
        let mut i = n as usize;
        while i > 0 {
            i -= 1;
            if t[i] < n {
                t[i] += 1;
                break;
            }
            t[i] = 1;
        }
    }
    set.into_iter().collect()
}

fn nb_distinctes(t: &[u8]) -> usize {
    let mut s = t.to_vec();
    s.sort();
    s.dedup();
    s.len()
}

fn tup(t: &[u8]) -> String {
    format!("({})", t.iter().map(|v| v.to_string()).collect::<Vec<_>>().join(", "))
}

fn swap_adjacent(t: &[u8], k: u8) -> Vec<u8> {
    let pk = t.iter().position(|v| *v == k).unwrap();
    let pk1 = t.iter().position(|v| *v == k + 1).unwrap();
    let mut l = t.to_vec();
    l.swap(pk, pk1);
    l
}

fn fusion(t: &[u8], k: u8) -> Vec<u8> {
    canonique(&t.iter().map(|v| if *v == k + 1 { k } else { *v }).collect::<Vec<u8>>())
}

fn next_comb(c: &mut Vec<usize>, n: usize) -> bool {
    let j = c.len();
    let mut i = j;
    while i > 0 {
        i -= 1;
        if c[i] < n - j + i {
            c[i] += 1;
            for k in i + 1..j {
                c[k] = c[k - 1] + 1;
            }
            return true;
        }
    }
    false
}

fn scissions(t: &[u8]) -> Vec<Vec<u8>> {
    let mut res: Vec<Vec<u8>> = vec![];
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    for v in vals {
        let p: Vec<usize> = t.iter().enumerate().filter(|(_, x)| **x == v).map(|(i, _)| i).collect();
        let m = p.len();
        if m < 2 {
            continue;
        }
        for j in 1..m {
            let mut c: Vec<usize> = (0..j).collect();
            loop {
                let s: BTreeSet<usize> = c.iter().map(|idx| p[*idx]).collect();
                let mut u = vec![0u8; t.len()];
                for (i, x) in t.iter().enumerate() {
                    u[i] = if *x == v {
                        if s.contains(&i) { v } else { v + 1 }
                    } else if *x > v {
                        x + 1
                    } else {
                        *x
                    };
                }
                res.push(canonique(&u));
                if !next_comb(&mut c, m) {
                    break;
                }
            }
        }
    }
    res
}

fn voisins_treillis(t: &[u8], n: u8) -> Vec<Vec<u8>> {
    let mut res: Vec<Vec<u8>> = vec![];
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    for k in 1..n {
        if vals.contains(&k) && vals.contains(&(k + 1)) {
            res.push(fusion(t, k));
        }
    }
    res.extend(scissions(t));
    res
}

fn bfs(src: &Vec<u8>, adj: &HashMap<Vec<u8>, Vec<Vec<u8>>>) -> HashMap<Vec<u8>, u32> {
    let mut dist: HashMap<Vec<u8>, u32> = HashMap::new();
    dist.insert(src.clone(), 0);
    let mut q: VecDeque<Vec<u8>> = VecDeque::new();
    q.push_back(src.clone());
    while let Some(u) = q.pop_front() {
        let du = dist[&u];
        for v in &adj[&u] {
            if !dist.contains_key(v) {
                dist.insert(v.clone(), du + 1);
                q.push_back(v.clone());
            }
        }
    }
    dist
}

fn construire_chemin(depart: &Vec<u8>, arrivee: &Vec<u8>, adj: &HashMap<Vec<u8>, Vec<Vec<u8>>>) -> Option<Vec<(Vec<u8>, Vec<u8>)>> {
    let mut parent: HashMap<Vec<u8>, Option<Vec<u8>>> = HashMap::new();
    parent.insert(depart.clone(), None);
    let mut q: VecDeque<Vec<u8>> = VecDeque::new();
    q.push_back(depart.clone());
    while let Some(u) = q.pop_front() {
        if u == *arrivee {
            break;
        }
        for v in &adj[&u] {
            if !parent.contains_key(v) {
                parent.insert(v.clone(), Some(u.clone()));
                q.push_back(v.clone());
            }
        }
    }
    if !parent.contains_key(arrivee) {
        return None;
    }
    let mut chaine: Vec<(Vec<u8>, Vec<u8>)> = vec![];
    let mut cur = arrivee.clone();
    while let Some(pp) = parent[&cur].clone() {
        chaine.push((pp.clone(), cur.clone()));
        cur = pp;
    }
    chaine.reverse();
    Some(chaine)
}

fn relabel(u: &[u8]) -> Vec<u8> {
    let mut rk = u.to_vec();
    rk.sort();
    rk.dedup();
    u.iter().map(|v| rk.iter().position(|w| w == v).unwrap() as u8 + 1).collect()
}

fn voisins_treillis_verif(t: &[u8]) -> Vec<Vec<u8>> {
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    let mut out: Vec<Vec<u8>> = vec![];
    for k in 1u8..6 {
        if vals.contains(&k) && vals.contains(&(k + 1)) {
            let merged: Vec<u8> = t.iter().map(|v| if *v == k + 1 { k } else { *v }).collect();
            out.push(relabel(&merged));
        }
    }
    for v in vals {
        let pos: Vec<usize> = t.iter().enumerate().filter(|(_, x)| **x == v).map(|(i, _)| i).collect();
        if pos.len() < 2 {
            continue;
        }
        for mask in 1..(1usize << pos.len()) - 1 {
            let bas: BTreeSet<usize> = (0..pos.len()).filter(|j| mask >> j & 1 == 1).map(|j| pos[j]).collect();
            let mut nouv = vec![0u8; 6];
            for i in 0..6 {
                nouv[i] = if bas.contains(&i) {
                    v
                } else if t[i] == v {
                    v + 1
                } else {
                    let x = t[i];
                    if x > v { x + 1 } else { x }
                };
            }
            out.push(relabel(&nouv));
        }
    }
    out
}

fn voisins_perm_verif(t: &[u8]) -> Vec<Vec<u8>> {
    let mut out: Vec<Vec<u8>> = vec![];
    for k in 1u8..6 {
        let u: Vec<u8> = t.iter().map(|v| if *v == k { v + 1 } else if *v == k + 1 { v - 1 } else { *v }).collect();
        out.push(u);
    }
    out
}

fn _bfs_depart(src: &Vec<u8>, voisins_fn: fn(&[u8]) -> Vec<Vec<u8>>) -> HashMap<Vec<u8>, u32> {
    let mut dist: HashMap<Vec<u8>, u32> = HashMap::new();
    dist.insert(src.clone(), 0);
    let mut q: VecDeque<Vec<u8>> = VecDeque::new();
    q.push_back(src.clone());
    while let Some(u) = q.pop_front() {
        let du = dist[&u];
        for v in voisins_fn(&u) {
            if !dist.contains_key(&v) {
                dist.insert(v.clone(), du + 1);
                q.push_back(v);
            }
        }
    }
    dist
}

fn verdict_chemin(depart: &Vec<u8>, arrivee: &Vec<u8>, chaine: &[(Vec<u8>, Vec<u8>)], voisins_fn: fn(&[u8]) -> Vec<Vec<u8>>, nom_modele: &str) -> String {
    if chaine.is_empty() {
        return "INVALIDE : chemin vide".to_string();
    }
    if chaine[0].0 != *depart || chaine[chaine.len() - 1].1 != *arrivee {
        return "INVALIDE : les extremites ne sont pas celles annoncees".to_string();
    }
    for (pred, cur) in chaine {
        if !voisins_fn(pred).iter().any(|v| *v == *cur) {
            return format!("INVALIDE : pas non elementaire {} -> {}", tup(pred), tup(cur));
        }
    }
    let dist = _bfs_depart(depart, voisins_fn);
    if chaine.len() as u32 != dist[arrivee] {
        return format!("INVALIDE : non minimal ({} pas, distance reelle {})", chaine.len(), dist[arrivee]);
    }
    format!("VALIDE [{}] : {} pas, minimalite certifiee par recherche exhaustive independante", nom_modele, chaine.len())
}

let identite: Vec<u8> = vec![1, 2, 3, 4, 5, 6];
let renverse: Vec<u8> = vec![6, 5, 4, 3, 2, 1];

let faux_compose: Vec<(Vec<u8>, Vec<u8>)> = vec![
    (identite.clone(), swap_adjacent(&swap_adjacent(&identite, 1), 2)),
    (swap_adjacent(&swap_adjacent(&identite, 1), 2), renverse.clone()),
];
let faux_extremites: Vec<(Vec<u8>, Vec<u8>)> = vec![(renverse.clone(), swap_adjacent(&renverse, 1))];

// Detour valide mais non minimal : le chemin de murs (10 pas) rallonge d'un aller-retour
// sur le mur 5~6 (fusion puis scission retour), toujours elementaire -> 12 pas.
let of6 = ordres_faibles(6);
let adj_c: HashMap<Vec<u8>, Vec<Vec<u8>>> = of6.iter().map(|t| (t.clone(), voisins_treillis(t, 6))).collect();
let cm = construire_chemin(&identite, &renverse, &adj_c).unwrap_or_else(Vec::new);
let x = fusion(&identite, 5);
let mut chemin_detour: Vec<(Vec<u8>, Vec<u8>)> = vec![(identite.clone(), x.clone()), (x, identite.clone())];
chemin_detour.extend(cm.iter().cloned());

println!("Controle 1 -- pas compose de deux swaps declare au verificateur du treillis :");
println!("  {}", verdict_chemin(&identite, &renverse, &faux_compose, voisins_treillis_verif, "treillis"));
println!("Controle 2 -- extremites fausses :");
println!("  {}", verdict_chemin(&identite, &renverse, &faux_extremites, voisins_perm_verif, "permutoedre"));
println!("Controle 3 -- detour de 12 pas (elementaire, extremites justes, non minimal) :");
println!("  {}", verdict_chemin(&identite, &renverse, &chemin_detour, voisins_treillis_verif, "treillis"));


Controle 1 -- pas compose de deux swaps declare au verificateur du treillis :


  INVALIDE : pas non elementaire (1, 2, 3, 4, 5, 6) -> (3, 1, 2, 4, 5, 6)


Controle 2 -- extremites fausses :


  INVALIDE : les extremites ne sont pas celles annoncees


Controle 3 -- detour de 12 pas (elementaire, extremites justes, non minimal) :


  INVALIDE : non minimal (12 pas, distance reelle 10)


### Lecture du résultat — le contrôle négatif : trois modes de refus, dont un détour de 12 pas jugé non minimal

Un vérificateur qui ne saurait que dire `VALIDE` ne certifierait rien. Les trois contrôles exhibent trois modes de refus distincts : le pas composé de deux swaps est rejeté comme **non élémentaire** (aucun voisinage ne contient ce saut double), les extrémités fausses sont détectées avant toute recherche, et — le contrôle qui fait le travail — un détour *parfaitement licite* de 12 pas (fusion vers le mur 5~6 puis scission retour, insérée dans le vrai chemin de 10) est détecté **non minimal**. C'est lui qui prouve que la preuve de minimalité a les dents : elle ne se contente pas de valider la connectivité, elle distingue 12 de 10.

In [13]:
// === Section 8 : geometriques identite -> renversement sur le permutoedre S6 ===

use std::collections::{BTreeSet, HashMap, VecDeque};

fn canonique(t: &[u8]) -> Vec<u8> {
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    t.iter().map(|v| vals.iter().position(|w| w == v).unwrap() as u8 + 1).collect()
}

fn ordres_faibles(n: u8) -> Vec<Vec<u8>> {
    let mut set = BTreeSet::new();
    let mut t = vec![1u8; n as usize];
    let total = (n as u64).pow(n as u32);
    for _ in 0..total {
        set.insert(canonique(&t));
        let mut i = n as usize;
        while i > 0 {
            i -= 1;
            if t[i] < n {
                t[i] += 1;
                break;
            }
            t[i] = 1;
        }
    }
    set.into_iter().collect()
}

fn nb_distinctes(t: &[u8]) -> usize {
    let mut s = t.to_vec();
    s.sort();
    s.dedup();
    s.len()
}

fn swap_adjacent(t: &[u8], k: u8) -> Vec<u8> {
    let pk = t.iter().position(|v| *v == k).unwrap();
    let pk1 = t.iter().position(|v| *v == k + 1).unwrap();
    let mut l = t.to_vec();
    l.swap(pk, pk1);
    l
}

fn fusion(t: &[u8], k: u8) -> Vec<u8> {
    canonique(&t.iter().map(|v| if *v == k + 1 { k } else { *v }).collect::<Vec<u8>>())
}

fn next_comb(c: &mut Vec<usize>, n: usize) -> bool {
    let j = c.len();
    let mut i = j;
    while i > 0 {
        i -= 1;
        if c[i] < n - j + i {
            c[i] += 1;
            for k in i + 1..j {
                c[k] = c[k - 1] + 1;
            }
            return true;
        }
    }
    false
}

fn scissions(t: &[u8]) -> Vec<Vec<u8>> {
    let mut res: Vec<Vec<u8>> = vec![];
    let mut vals = t.to_vec();
    vals.sort();
    vals.dedup();
    for v in vals {
        let p: Vec<usize> = t.iter().enumerate().filter(|(_, x)| **x == v).map(|(i, _)| i).collect();
        let m = p.len();
        if m < 2 {
            continue;
        }
        for j in 1..m {
            let mut c: Vec<usize> = (0..j).collect();
            loop {
                let s: BTreeSet<usize> = c.iter().map(|idx| p[*idx]).collect();
                let mut u = vec![0u8; t.len()];
                for (i, x) in t.iter().enumerate() {
                    u[i] = if *x == v {
                        if s.contains(&i) { v } else { v + 1 }
                    } else if *x > v {
                        x + 1
                    } else {
                        *x
                    };
                }
                res.push(canonique(&u));
                if !next_comb(&mut c, m) {
                    break;
                }
            }
        }
    }
    res
}

fn voisins_permutoedre(t: &[u8], n: u8) -> Vec<Vec<u8>> {
    (1..n).map(|k| swap_adjacent(t, k)).collect()
}

fn bfs(src: &Vec<u8>, adj: &HashMap<Vec<u8>, Vec<Vec<u8>>>) -> HashMap<Vec<u8>, u32> {
    let mut dist: HashMap<Vec<u8>, u32> = HashMap::new();
    dist.insert(src.clone(), 0);
    let mut q: VecDeque<Vec<u8>> = VecDeque::new();
    q.push_back(src.clone());
    while let Some(u) = q.pop_front() {
        let du = dist[&u];
        for v in &adj[&u] {
            if !dist.contains_key(v) {
                dist.insert(v.clone(), du + 1);
                q.push_back(v.clone());
            }
        }
    }
    dist
}

fn compter_geodesiques(depart: &Vec<u8>, arrivee: &Vec<u8>, adj: &HashMap<Vec<u8>, Vec<Vec<u8>>>) -> u64 {
    // Nombre de chemins minimaux : produits des chemins entrants, couche par couche (DAG des couches).
    let dist = bfs(depart, adj);
    let d = dist[arrivee];
    let mut nb: HashMap<&Vec<u8>, u64> = HashMap::new();
    nb.insert(depart, 1);
    for couche in 1..=d {
        let cibles: Vec<&Vec<u8>> = dist.iter().filter(|(_, du)| **du == couche).map(|(k, _)| k).collect();
        for u in cibles {
            let mut s: u64 = 0;
            for v in &adj[u] {
                if dist.get(v) == Some(&(couche - 1)) {
                    s += nb[v];
                }
            }
            nb.insert(u, s);
        }
    }
    nb[arrivee]
}

let of6 = ordres_faibles(6);
let st6: Vec<Vec<u8>> = of6.iter().filter(|t| nb_distinctes(t) == 6).cloned().collect();
let adj_p: HashMap<Vec<u8>, Vec<Vec<u8>>> = st6.iter().map(|t| (t.clone(), voisins_permutoedre(t, 6))).collect();

let identite: Vec<u8> = vec![1, 2, 3, 4, 5, 6];
let renverse: Vec<u8> = vec![6, 5, 4, 3, 2, 1];
let geo6 = compter_geodesiques(&identite, &renverse, &adj_p);
println!("Geodesiques (1,2,3,4,5,6) -> (6,5,4,3,2,1) sur une table 3x2 : {}", geo6);
println!("Rappel GT-20 (S4, substrat 2x2) : 16 geodesiques pour l'antipode");


Geodesiques (1,2,3,4,5,6) -> (6,5,4,3,2,1) sur une table 3x2 : 292864


Rappel GT-20 (S4, substrat 2x2) : 16 geodesiques pour l'antipode


### Lecture du résultat — 292 864 géodésiques au lieu de 16 : l'explosion du nombre de chemins minimaux

L'explosion du nombre de géodésiques (16 sur S4, **292 864** sur S6 — la valeur mesurée ci-dessus) matérialise la richesse du nouveau substrat : il n'existe pas *un* chemin minimal mais une famille entière, et le diamètre 15 n'est réalisé que par une fraction organisée des 720 tables. Ce comptage par DAG des couches est le même instrument que GT-20 (cross-check direct avec ses 16), porté à l'échelle supérieure sans changement de méthode — la reproductibilité de l'instrument à travers les substrats fait partie de l'attestation.

Le nombre de géodésiques dit aussi quelque chose de la *robustesse* du témoin du §6 : si l'antipode n'était atteint que par un unique chemin de 10 pas, on pourrait soupçonner une coincidence structurelle ; le fait qu'il existe par ailleurs des familles entières de chemins minimaux dans chaque modèle rappelle que la minimalité certifiée par le vérificateur porte sur la *longueur*, et que le constructeur n'en a extrait qu'un représentant. Un prolongement naturel (hors scope ici) serait de compter les géodésiques du trellis entre antipodes — la version « à travers les murs » de ce comptage — pour comparer les deux cardinaux.

In [14]:
// === Exercice 1 : chemin retour Renversement -> Identite, construit puis verifie independamment.
// Etape 1 : chemin_retour = construire_chemin(RENVERSE[0], IDENTITE[0], graphs[6][1])
// Etape 2 : verdict = verifier_chemin(RENVERSE[0], IDENTITE[0], chemin_retour)
// Indice : le treillis est non dirige -- la longueur du retour doit egaler 10.
let _chemin_retour: Option<Vec<(Vec<u8>, Vec<u8>)>> = None; // TODO etudiant
let _verdict_retour: Option<String> = None; // TODO etudiant
println!("Exercice 1 : a completer (chemin retour Renversement -> Identite par les murs)");


Exercice 1 : a completer (chemin retour Renversement -> Identite par les murs)


### Exercice 1 — Le chemin retour

Construisez le chemin de retour `Renversement -> Identité` sur le graphe du treillis, puis faites-le certifier par le vérificateur. *Indice :* le graphe est non dirigé, donc la longueur du retour est connue d'avance — c'est exactement ce que le vérificateur doit confirmer par sa propre recherche exhaustive.

In [15]:
// === Exercice 2 : les antipodes raccourcissent-ils tous ?
// Etape 1 : dperm = bfs(IDENTITE[0], graphs[6][0]) ; antipodes = [t for t in ST6 if dperm[t] == 15]
// Etape 2 : dtl = bfs(IDENTITE[0], graphs[6][1]) ; compter combien d'antipodes ont dtl[t] < 15.
// Indice : le chemin tout-lie (5 fusions + 5 scissions = 10) n'est pas la seule facon de gagner.
let _antipodes_raccourcis: Option<usize> = None; // TODO etudiant
println!("Exercice 2 : a completer (combien des antipodes de distance 15 sont raccourcis par les murs)");


Exercice 2 : a completer (combien des antipodes de distance 15 sont raccourcis par les murs)


### Exercice 2 — Le recensement des antipodes

Parmi les tables situées à la distance maximale 15 de l'identité dans le permutoèdre (les antipodes), combien sont raccourcies par les murs ? *Indice :* le chemin tout-lié en vaut 5 d'économie, mais certaines réorganisations partielles peuvent faire mieux ou moins bien — la distribution des gains est le sujet.

In [16]:
// Exercice 3 : traverser un mur triple.
// Etape 1 : cible = canonique((2, 2, 2, 1, 3, 4))  // trois cases liees au rang 2
// Etape 2 : chemin = construire_chemin(IDENTITE[0], cible, graphs[6][1])
// Etape 3 : verdict = verifier_chemin(IDENTITE[0], cible, chemin)
// Indice : combien de fusions pour lier trois cases, et la scission finale est-elle utile ici ?
let _mur_triple: Option<Vec<(Vec<u8>, Vec<u8>)>> = None; // TODO etudiant
println!("Exercice 3 : a completer (chemin minimal vers une table portant un ex aequo triple)");


Exercice 3 : a completer (chemin minimal vers une table portant un ex aequo triple)


### Exercice 3 — Le mur triple

Construisez et certifiez un chemin minimal de l'identité vers une table portant un ex æquo **triple**. *Indice :* lier trois cases demande deux fusions ; demandez-vous si le vérificateur accepte un chemin qui *re-descend* scissionner quelque part — la réponse mesure la frontière entre réordonner par les murs et payer les allers-retours.

## Conclusion — ce que cette seconde attestation établit

**Pour la table des opérations [#12204](https://github.com/jsboige/CoursIA/issues/12204), opération 13 « Traverser un mur »** : le témoin GT-20 (2×2) n'était pas isolé dans son genre — sa méthode (constructeur/vérificateur séparés, recensement exhaustif, contrôle négatif) se transpose à un second substrat, et c'est précisément cette transplantation qui produit la connaissance :

1. **Le verdict 2×2 est robuste au modèle fermé** : 0 raccourci sur 576 paires, murs traversables dans les deux sens (§3). La mesure GT-20 n'était pas un artefact du sens unique.
2. **Le verdict ne survit pas au substrat 3×2** : 77 760 raccourcis sur 518 400 paires (15,0 %), diamètre par côté contracté de 15 à 10 (§4). Traverser les murs y est un raccourci *structurel* — porté par les scissions de blocs — et non une curiosité marginale.
3. **Le témoin est certifié** : chemin de l'antipode à travers le mur total (5 fusions + 5 scissions), construit puis validé par un vérificateur qui re-dérive tout (§6), contrôle négatif à l'appui (§7).

**Résiduel assumé.** L'univers complet de jeux avec murs bilatéraux (≈ 21,9 M sommets) est couvert par le lemme de décomposition (§5), contrôlé mais pas prouvé formellement ici ; une preuve Lean du lemme serait le prolongement naturel côté `game_theory_lean`.

**Généalogie** : [Metajeux-04](Metajeux-04-Chemin-Minimal-Robinson-Goforth-Python.ipynb) (témoin constructeur/vérificateur, 2×2) · [Metajeux-05](Metajeux-05-Temoins-Impossibilite-Python.ipynb) (témoins d'impossibilité) · ce notebook (second substrat, verdict de frontière).

**Retour à la série.** [GameTheory — README](../README.md) — Metajeux-06 est le dernier carnet de la sous-série `Metajeux`.

## À retenir

- **Le verdict 2×2 ne survit PAS au substrat 3×2** : ici, les murs *raccourcissent* — `77 760` raccourcis sur `518 400` paires (`15.0%`), et le diamètre par côté est contracté de 15 à 10. Traverser les murs y est un raccourci *structurel* (porté par les scissions de blocs), pas une curiosité marginale.
- **Le témoin est certifié par transplantation de méthode** : le constructeur produit le chemin de l'antipode à travers le mur total (5 fusions + 5 scissions), et un vérificateur *re-dérive tout* pour attester sa minimalité — le même instrument que GT-20, porté à l'échelle supérieure.
- **La richesse du nouveau substrat s'explore en nombre de chemins minimaux** : `292 864` géodésiques pour l'antipode 3×2, contre 16 sur `S₄` — le comptage par DAG des couches est le même instrument, sans changement de méthode.

## Vérifiez votre compréhension

1. Pourquoi le verdict « 0 raccourci » de GT-20 (2×2) ne se transpose-t-il PAS au substrat 3×2 ? Qu'est-ce que les scissions de blocs changent à la géométrie ?
2. En quoi le comptage `292 864` (contre 16) atteste-t-il de la *robustesse* du témoin plutôt que d'une coïncidence structurelle ?
3. Que couvre le lemme de décomposition (§5) pour l'univers complet à murs bilatéraux (≈ 21,9 M sommets), et pourquoi sa preuve Lean reste-t-elle un résiduel assumé ?

<details>
<summary>Éléments de réponse</summary>

1. Parce que le troc n'a pas la même arithmétique aux deux tailles. Une traversée de mur coûte au moins deux pas — une fusion, une scission — et ne peut être rentable que si le déplacement de blocs économise au moins deux swaps ; renverser n cases coûte `C(n,2)` swaps au permutoèdre contre `2(n-1)` pas par les murs, d'où un gain net nul à 4 cases et de 5 à 6 cases. À 2×2, la mesure le confirme : `0` raccourci sur `576` paires, `48` égalités. À 3×2, une seule scission d'un bloc de taille m replace m cases d'un coup là où le permutoèdre les permute une à une : le recensement exhaustif donne `77760` raccourcis sur `518400` paires (`15.0 %`), le chemin de l'antipode passant de `15` swaps à `10` pas — les cinq premières étapes fusionnent, les cinq suivantes scissionnent — et le diamètre par côté se contractant de `15` à `10`. La scission change donc la géométrie : elle rend le déplacement massif possible en un pas, ce qui transforme une stratification métriquement neutre en compression métrique.
2. Parce que le même instrument redonne un compte sur le nouveau substrat sans changer de méthode : `292864` géodésiques pour l'antipode sur une table 3×2, contre `16` sur `S₄` rappelé dans la même sortie. Le témoin du §6 n'est donc pas un chemin unique qu'un accident aurait produit : sur le permutoèdre, l'antipode est atteint par une famille entière de chemins minimaux, et la propriété certifiée porte sur la **longueur** — le vérificateur re-dérive la distance exhaustive et rend `VALIDE [permutoedre] : 15 pas` et `VALIDE [treillis] : 10 pas`, et le contrôle négatif distingue `12` de `10` —, pas sur l'unicité de l'itinéraire. Le constructeur n'extrait qu'un représentant de la famille ; ce qui rend le témoin robuste, c'est que la minimalité vaut pour toute la famille et que l'instrument de comptage se reproduit d'un substrat à l'autre. Le diamètre mesuré `15` rejoint d'ailleurs la prédiction de Coxeter (inversions de `w₀` dans `S₆`), ce qui écarte l'hypothèse d'une coïncidence du substrat.
3. Le lemme `d_jeu(G,H) = d_cote(row_G,row_H) + d_cote(col_G,col_H)`, valable dans les deux modèles, couvre l'univers des jeux à murs des deux côtés — `4683` ordres faibles par côté, soit `4683² ≈ 21,9 millions` de sommets — sans l'énumérer : chaque générateur ne touche qu'un seul côté, donc toute distance de jeu se décompose en deux distances de côté. C'est ce qui rend le recensement du §4, mené sur les seules paires de chambres, exhaustif **pour les jeux** au sens du lemme. Le contrôle direct existe : la BFS sur le produit des `518400` chambres atteint `518400` jeux et donne, identité → renversement, `directe = 30 | par decomposition = 30`, `concordance : True` — mais un contrôle n'est pas une preuve ; la preuve est l'argument « un pas touche un seul côté », contrôlé là où une erreur serait maximale (l'antipode, distance `30`, le diamètre). Sa formalisation Lean n'est pas faite ici : le carnet l'assume comme résiduel et la renvoie au prolongement naturel côté `game_theory_lean`.

</details>

## Pour aller plus loin

- **La généalogie de la série** : [Metajeux-04](Metajeux-04-Chemin-Minimal-Robinson-Goforth-Python.ipynb) (témoin 2×2, murs non-raccourcissants) · [Metajeux-05](Metajeux-05-Temoins-Impossibilite-Python.ipynb) (témoins d'impossibilité) · ce notebook (second substrat, murs raccourcissants).
- **Le lemme de décomposition en Lean** : le prolongement naturel côté `game_theory_lean` pour prouver formellement la couverture de l'univers complet.